# (IIP314W-2) [2026-T2] OPTIMIZACIÓN APLICADA A NEGOCIOS
## Ayudantía 4: Modelamiento matemático de programación lineal

---

**Profesor:** Rodrigo Trigo Vilches

**Ayudante:** Vicente Ramírez

**Fecha:** 24 de junio de 2026

**Universidad del Desarrollo**

---

## Objetivos de la ayudantía

- Traducir un problema de negocios a un modelo de **programación lineal**: conjuntos, parámetros, variables de decisión, función objetivo y restricciones.
- Escribir el modelo de forma **algebraica** (paramétrica, en notación de sumatorias), **sin valores numéricos**: separar la *estructura* del modelo de los *datos*.
- Distinguir los tipos de restricciones típicas: **capacidad** ($\le$), **requerimiento/cobertura** ($\ge$) y **logística**.
- Implementar el modelo **genéricamente** en `gurobipy`, leyendo los datos desde archivos **CSV**.
- Interpretar la solución y los **precios sombra** (`.Pi`) de los recursos, conectando con la Ayudantía 3 (KKT).

## Repaso: elementos de un modelo (Clase 9)

| Elemento | Qué es |
|:--|:--|
| **Conjuntos / índices** | sobre qué corren las variables y restricciones (productos, plantas, misiones, …) |
| **Parámetros** | datos conocidos: costos, capacidades, demandas, precios (los escribimos con **letras**, no números) |
| **Variables de decisión** | lo que se busca |
| **Función objetivo** | qué se minimiza o maximiza |
| **Restricciones** | lo que debe cumplirse |

**Sobre los tipos de variable (importante).** En las Clases 9–10, **todas** las variables de decisión son **continuas** ($x \ge 0$). Las variables **enteras** solo se mencionaron al pasar (los manteles, Clase 10) sin desarrollar un método; y las variables **binarias**, las **restricciones de activación** (relación binaria–continua) y la técnica **"big-M"** **no se han visto en clase**. Por lo tanto, en esta ayudantía usamos **únicamente variables continuas no negativas**.

**Tipos de restricción que usaremos:**
- **Capacidad / disponibilidad** ($\sum \le b$): un recurso que no se puede sobrepasar.
- **Requerimiento / cobertura** ($\sum \ge d$): una exigencia mínima que cumplir.
- **Logística** ($\sum \le$ capacidad): transporte o almacenamiento que limita el flujo.

## Metodología: modelo algebraico + datos en CSV

Para que el modelo sea **100% algebraico**, los números **no** se escriben en el notebook. Se generan con el script **`generar_datos_ayudantia4.py`** (semilla fija, con **factibilidad garantizada**) y se guardan como **CSV** en la carpeta `datos_ayudantia4/`. El modelo en `gurobipy` **lee** esos CSV y se arma con `addVars` y `quicksum` sobre los conjuntos — sin números "a mano".

> Para (re)generar los datos, ejecutar en la terminal:
> ```
> py -3.13 generar_datos_ayudantia4.py
> ```
> Cambiar la semilla `SEED` dentro del script produce otra instancia (también factible) sin tocar el modelo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB

DATA = "datos_ayudantia4"   # carpeta con los CSV generados por el script

## Ejercicio 1 — Planificación de entrenamiento militar (minimizar costos)

Una base de la **Fuerza Aérea** debe planificar, al **mínimo costo**, las **horas de vuelo de entrenamiento** del próximo trimestre para mantener certificadas a sus tripulaciones. La base opera un conjunto de aeronaves $\mathcal{A}$ —un caza, un helicóptero y un avión de transporte— y debe acumular horas en un conjunto de misiones $\mathcal{M}$: navegación, tiro real y asalto. No toda aeronave puede ejecutar toda misión (el transporte, por ejemplo, no realiza tiro), de modo que solo se consideran los pares **compatibles** $(i,j)\in\mathcal{C}$.

Volar una hora la aeronave $i$ tiene tres componentes de costo. Primero, el **combustible**: cada aeronave consume $f_i$ litros por hora de vuelo, y el litro cuesta $p$ (el mismo precio para todas). Segundo, la **operación y mantención**, $o_i$ por hora de la aeronave $i$. Tercero, en las misiones de tiro se gastan **municiones**, con un costo de $a_{ij}$ por hora (cero en las misiones que no disparan). Así, el costo por hora de cada par compatible resulta $\;c_{ij} = f_i\,p + o_i + a_{ij}$.

Las exigencias del plan son las siguientes. Cada misión $j$ requiere acumular al menos $r_j$ horas para certificar a las tripulaciones. Cada aeronave $i$ tiene, por mantenimiento y vida útil, un tope de $H_i$ horas de vuelo en el trimestre; y, para no perder la habilitación de sus pilotos, debe volar **al menos** $\underline{H}_i$ horas (este piso es relevante para el caza; para las demás aeronaves es 0). La logística de **combustible** también limita: el trimestre dispone de $Q$ litros en total. Por último, las **municiones** son finitas: cada hora de tiro de la aeronave $i$ consume $m_i$ proyectiles, y el stock disponible es de $S$ unidades.

**Tareas:** (1) definir las variables; (2) plantear FO y restricciones **algebraicamente**; (3) implementar en `gurobipy` leyendo los CSV y resolver; (4) identificar qué recursos son el **cuello de botella**.

### Formulación

_Plantee el modelo de forma **algebraica** (notación de sumatorias, con los parámetros simbólicos del enunciado)._

In [ ]:
# Cargar los datos del Ejercicio 1 desde los CSV
aero = pd.read_csv(f"{DATA}/mil_aeronaves.csv")   # aeronave, f, o, H, m, Hmin
mis  = pd.read_csv(f"{DATA}/mil_misiones.csv")     # mision, r
comp = pd.read_csv(f"{DATA}/mil_compat.csv")       # aeronave, mision, a  (pares compatibles)
esc  = pd.read_csv(f"{DATA}/mil_escalares.csv").set_index("parametro")["valor"]

display(aero); display(mis); display(comp); print(esc)

In [ ]:
# Conjuntos y parametros ya estan en los DataFrames cargados (aero, mis, comp, esc).
# Arme los diccionarios de parametros, cree el modelo, las variables x sobre 'pares',
# la funcion objetivo (min costo) y las 5 familias de restricciones.
#### CÓDIGO AQUÍ ####
# Pistas:
#   A = list(aero["aeronave"]); M = list(mis["mision"]); pares = list(zip(comp.aeronave, comp.mision))
#   x = mdl.addVars(pares, lb=0)               # continuas
#   c[i,j] = f[i]*p + o[i] + a[i,j]            # min sum c_ij x_ij
#   req_j : sum_i x[i,j] >= r[j]      |  disp_i: sum_j x[i,j] <= H[i]
#   comb  : sum f[i]*x[i,j] <= Q      |  muni  : sum m[i]*x[i,Tiro] <= S
#   min_i : sum_j x[i,j] >= Hmin[i]   (solo si Hmin[i] > 0)


In [ ]:
# Muestre el plan optimo de horas por (aeronave, mision) y determine que restricciones
# quedan ACTIVAS (cuello de botella): requerimientos, disponibilidad, combustible, municiones, caza minima.
#### CÓDIGO AQUÍ ####


In [ ]:
# (Opcional) Grafique el plan: horas por aeronave desglosadas por mision (barra apilada).
#### CÓDIGO AQUÍ ####


### Discusión

_Responda: ¿qué aeronave concentra las horas y por qué? ¿Por qué vuela el caza si es la más cara? ¿Qué restricciones quedan **activas** (cuello de botella) y cuáles con holgura? ¿Dónde convendría invertir según eso?_

_Respuesta:_ 

## Ejercicio 2 — Distribución de una empresa de helados (maximizar ingresos)

Una **empresa de helados** quiere planificar su distribución para **maximizar los ingresos** de la temporada. Elabora un conjunto de productos $\mathcal{P}$ —una línea premium y una clásica— en un conjunto de plantas $\mathcal{K}$, y los vende en un conjunto de mercados $\mathcal{I}$. La decisión es cuántas **cajas** de cada producto despachar desde cada planta hacia cada mercado.

El precio de venta depende del producto y del mercado: una caja del producto $p$ se vende a $\pi_{pi}$ en el mercado $i$ (la línea premium se paga más, y algunos mercados pagan mejor que otros). Como interesa la **facturación** y no la utilidad, **no se descuentan costos** en la función objetivo; el costo logístico aparece, en cambio, como una **restricción de capacidad**.

Del lado de la **oferta**, cada planta $k$ puede producir a lo más $C_{pk}$ cajas del producto $p$. Del lado de la **demanda**, el mercado $i$ no absorbe más de $D_{pi}$ cajas del producto $p$ (despachar por sobre ese tope no genera ingreso). Y en el medio está la **cadena de frío**: todo viaja en camiones refrigerados, cada caja del producto $p$ ocupa $v_p$ metros cúbicos, y la flota refrigerada asignada a la planta $k$ tiene una capacidad total de $L_k$ m³ por temporada.

**Tareas:** (1) definir variables; (2) formular FO y restricciones **algebraicamente**; (3) implementar leyendo los CSV y resolver; (4) interpretar los **precios sombra** (`.Pi`) de la flota y de la demanda (conexión con la Ayudantía 3).

### Formulación

_Plantee el modelo de forma **algebraica** (con los parámetros simbólicos del enunciado)._

In [ ]:
# Cargar los datos del Ejercicio 2 desde los CSV
prec = pd.read_csv(f"{DATA}/helados_precios.csv")     # producto, mercado, precio
prod = pd.read_csv(f"{DATA}/helados_produccion.csv")  # producto, planta, capacidad
dem  = pd.read_csv(f"{DATA}/helados_demanda.csv")      # producto, mercado, demanda
vol  = pd.read_csv(f"{DATA}/helados_volumen.csv")      # producto, volumen
flo  = pd.read_csv(f"{DATA}/helados_flota.csv")        # planta, capacidad_m3

display(prec); display(prod); display(dem); display(vol); display(flo)

In [ ]:
# Conjuntos y parametros ya estan en los DataFrames cargados (prec, prod, dem, vol, flo).
# Cree el modelo, las variables x[p,k,i], la FO (MAX ingresos) y las 3 familias de restricciones.
#### CÓDIGO AQUÍ ####
# Pistas:
#   P = list(vol["producto"]); K = list(flo["planta"]); I = list(dict.fromkeys(prec["mercado"]))
#   x = mdl.addVars(P, K, I, lb=0)               # continuas
#   max  sum_{p,k,i} pi[p,i] * x[p,k,i]
#   prod : sum_i x[p,k,i] <= C[p,k]              (por producto y planta)
#   dem  : sum_k x[p,k,i] <= D[p,i]              (por producto y mercado)
#   flota: sum_{p,i} v[p]*x[p,k,i] <= L[k]       (por planta)


In [ ]:
# Muestre el plan de despacho (cajas por producto-planta-mercado) e interprete los
# precios sombra .Pi de la flota (floC[k].Pi) y de la demanda (demC[(p,i)].Pi).
#### CÓDIGO AQUÍ ####


In [ ]:
# (Opcional) Grafique los ingresos por mercado (barra).
#### CÓDIGO AQUÍ ####


### Discusión

_Responda: ¿por qué el costo logístico no está en la función objetivo? ¿Qué producto se prioriza y por qué (pista: ingreso por m³)? ¿Qué mide el `.Pi` de la flota y el de la demanda? ¿Cómo se conecta con los multiplicadores KKT de la Ayudantía 3?_

_Respuesta:_ 